# ProphetGP 사용 예시

이 노트북은 ProphetGP의 핵심 기능(학습, 후보 추천, 데이터셋 추가)을 빠르게 실행해보는 예시입니다.

In [1]:
# 필요 시 주석 해제 후 설치
# !pip install -e .[dev]

In [ ]:
import numpy as np
from pathlib import Path

from prophet_gp.config import load_config
from prophet_gp.pipeline.trainer import ProphetGPPipeline
from prophet_gp.data.dataset import ReactionDatasetService

# 노트북 실행 위치가 notebooks/여도 안전하게 프로젝트 루트를 찾는다.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "test_20260530.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "sample" / "sample_data_extended1.csv"
NEW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "new_batch.csv"
MERGED_OUT_PATH = PROJECT_ROOT / "data" / "raw" / "reactions_merged.csv"

config = load_config(CONFIG_PATH)
pipeline = ProphetGPPipeline(config)
dataset_service = ReactionDatasetService(config.data)

print("Project root:", PROJECT_ROOT)
print("Config loaded:", config.model_dump())

Project root: c:\Projects\ProphetGP
Config loaded: {'data': {'reactant_column': 'Mol.1', 'target_column': ['Emission Peak', 'FWHM'], 'reactant_delimiter': '|', 'ignore_columns': ['PLQY', 'Pristine Emission', 'Pristine FWHM', 'Pristine PLQY'], 'reactant_allowed_values': ['1,5-Diaminonaphthalene', '1,8-Diaminonaphthalene', '2,3-Diaminonaphthalene', '2,6-Diaminonaphthalene', '1,2-Diaminonaphthalene', '1,3-Diaminonaphthalene', '1,4-Diaminonaphthalene', '1,6-Diaminonaphthalene', '1,7-Diaminonaphthalene', '2,7-Diaminonaphthalene'], 'condition_ranges': {'Temperature': {'min': 0.0, 'max': 600.0, 'allowed_values': None}}, 'explicit_condition_types': {'Temperature': 'continuous'}}, 'featurization': {'featuriser': 'morgan_fp', 'combine_strategy': 'concat'}, 'optimization': {'objective': 'target', 'suggestion_strategy': 'best_output', 'standardize_gp_inputs': False, 'standardize_gp_targets': False, 'target_value': None, 'target_objectives': {'Emission Peak': {'objective': 'target', 'target_value':

In [2]:
# 1) 사용 가능한 featuriser 확인
available_featurisers = pipeline.featurizers.available()
print("Available featurisers count:", len(available_featurisers))
print(available_featurisers[:20])  # 앞쪽 일부만 출력

Available featurisers count: 10
['bag_of_characters', 'drfp', 'ecfp_fingerprints', 'fragments', 'molecular_graphs', 'morgan_fp', 'mqn_features', 'one_hot', 'rdkit_descriptors', 'rxnfp']


In [3]:
# 2) 학습
artifacts = pipeline.train_from_csv(DATA_PATH)
print("Train rows:", artifacts.x_train.shape[0])
print("Feature dims:", artifacts.x_train.shape[1])
print("molecular representation check:", np.unique(artifacts.x_train[:, :-1], axis=0).shape)

c:\Users\sung1234\AppData\Local\Programs\Python\Python39\lib\site-packages\botorch\models\utils\assorted.py:174: InputDataWarning: Input data is not contained to the unit cube. Please consider min-max scaling the input data.
  warnings.warn(msg, InputDataWarning)
c:\Users\sung1234\AppData\Local\Programs\Python\Python39\lib\site-packages\botorch\models\utils\assorted.py:202: InputDataWarning: Input data is not standardized (mean = tensor([505.4545], dtype=torch.float64), std = tensor([40.0366], dtype=torch.float64)). Please consider scaling the input to zero mean and unit variance.
  warnings.warn(msg, InputDataWarning)
c:\Users\sung1234\AppData\Local\Programs\Python\Python39\lib\site-packages\botorch\models\utils\assorted.py:174: InputDataWarning: Input data is not contained to the unit cube. Please consider min-max scaling the input data.
  warnings.warn(msg, InputDataWarning)
c:\Users\sung1234\AppData\Local\Programs\Python\Python39\lib\site-packages\botorch\models\utils\assorted.py:2

Train rows: 22
Feature dims: 1025
molecular representation check: (6, 1024)


In [33]:
# 3) 다음 실험 조건 후보 추천 (raw + 해석 결과)
# strategy: "best_output" | "best_information"
n_candidates = 3
strategy = "best_output"
suggestions = pipeline.suggest_next_experiments(
    artifacts,
    n_candidates=n_candidates,
    strategy=strategy,
)

print("Strategy:", strategy)
print("Raw candidates shape:", suggestions.raw_candidates.shape)
print("Decoded candidates:")
for idx, row in enumerate(suggestions.decoded_candidates, 1):
    print(f"- candidate_{idx}")
    print("  predicted mean:", row["predicted_target_mean"])
    print("  predicted std:", row["predicted_target_std"])
    print("  target gap:", row["target_gap"])
    print("  nearest_known_reactants_input:", row["nearest_known_reactants_input"])
    print("  Temperature:", row["Temperature"])
    print("  mapped input:", row)

# suggestions.raw_candidates

Strategy: best_output
Raw candidates shape: (3, 2051)
Decoded candidates:
- candidate_1
  predicted mean: {'Emission Peak': 499.11926745725884, 'FWHM': 104.8125588557724}
  predicted std: {'Emission Peak': 17.4134578988996, 'FWHM': 3.226394056243256}
  target gap: {'Emission Peak': 9.119267457258843, 'FWHM': None}
  nearest_known_reactants_input: 2,6-Diaminonaphthalene
  Temperature: 275.4651184082031
  mapped input: {'predicted_target_mean': {'Emission Peak': 499.11926745725884, 'FWHM': 104.8125588557724}, 'predicted_target_std': {'Emission Peak': 17.4134578988996, 'FWHM': 3.226394056243256}, 'target_gap': {'Emission Peak': 9.119267457258843, 'FWHM': None}, 'objective_score': -92.96931454187677, 'information_score': 19.994573143894208, 'total_score': -92.96931454187677, 'ranking_strategy': 'best_output', 'mapped_reactants_input': '2,6-Diaminonaphthalene', 'mapped_reactants_smiles': ['Nc1ccc2cc(N)ccc2c1'], 'nearest_known_reactants_input': '2,6-Diaminonaphthalene', 'nearest_known_reacta

In [ ]:
# 4) 신규 배치 데이터 append
# 파일이 준비되어 있지 않으면 이 셀은 건너뛰세요.
merged = dataset_service.append_csv(DATA_PATH, NEW_DATA_PATH, MERGED_OUT_PATH)
print("Merged rows:", len(merged))
print("Saved to:", MERGED_OUT_PATH)

## 입력 데이터 포맷 가이드

- `reactants`: 반응물 리스트 (`|` 구분)
- `target` 또는 `target` 리스트: 예측/최적화 대상 물성값(단일/다중 타깃 모두 지원)
- 그 외 컬럼: 반응 조건(문자열/숫자 모두 가능, 자동 타입 추론 + config override 지원)